# Practical 8: Foundational Data Manipulation Using Pandas

**CampusCare — Student Performance Analytics**

This notebook builds basic Pandas skills on a small, fictional student performance dataset. The same file later feeds the Student Performance Analytics section on the CampusCare home page.

**Pipeline:** `data/student_performance.csv` → Pandas cleaning and analysis → `data/student_performance_processed.json` → landing page.

The raw file intentionally includes a few missing values and one duplicate row so the cleaning steps have something real to fix.


## 1. Import Pandas

Pandas is the library used to load the CSV, clean it, and calculate simple academic summaries.


In [1]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Find the project folder whether Jupyter was started from CampusCare/ or notebooks/.
cwd = Path.cwd()
if (cwd / "data" / "student_performance.csv").exists():
    root = cwd
elif (cwd.parent / "data" / "student_performance.csv").exists():
    root = cwd.parent
else:
    raise FileNotFoundError("Open this notebook from the CampusCare project.")

csv_path = root / "data" / "student_performance.csv"
print("Dataset:", csv_path)


Dataset: /Users/hirenrathva/Desktop/CampusCare/data/student_performance.csv


## 2. Load the Dataset

`pd.read_csv()` reads the student performance file into a DataFrame. A DataFrame is a table with rows and named columns.


In [2]:
df = pd.read_csv(csv_path)
df


,student_id,student_name,department,attendance,study_hours,assignments_completed,internal_marks,final_marks
0,STU001,Aarav Shah,Computer Science,92.0,5.5,8.0,38,86.0
1,STU002,Diya Patel,Computer Science,88.0,4.5,8.0,36,82.0
2,STU003,Kabir Mehta,Computer Science,95.0,6.0,9.0,40,91.0
3,STU004,Ananya Iyer,Computer Science,81.0,3.5,7.0,32,74.0
4,STU005,Rohan Desai,Computer Science,76.0,3.0,6.0,30,68.0
5,STU006,Meera Nair,Computer Science,90.0,5.0,8.0,37,85.0
6,STU007,Vivaan Joshi,Computer Science,84.0,4.0,7.0,34,78.0
7,STU008,Sara Khan,Computer Science,70.0,2.5,5.0,28,61.0
8,STU009,Arjun Reddy,Computer Science,97.0,6.5,9.0,39,94.0
9,STU010,Ishita Gupta,Computer Science,79.0,3.5,7.0,33,72.0


## 3. Display the Dataset

`head()` shows the first five rows so we can see what the columns look like. `tail()` shows the last five rows, including the extra copy of a student that was left in the raw file.


In [3]:
df.head()


,student_id,student_name,department,attendance,study_hours,assignments_completed,internal_marks,final_marks
0,STU001,Aarav Shah,Computer Science,92.0,5.5,8.0,38,86.0
1,STU002,Diya Patel,Computer Science,88.0,4.5,8.0,36,82.0
2,STU003,Kabir Mehta,Computer Science,95.0,6.0,9.0,40,91.0
3,STU004,Ananya Iyer,Computer Science,81.0,3.5,7.0,32,74.0
4,STU005,Rohan Desai,Computer Science,76.0,3.0,6.0,30,68.0


In [4]:
df.tail()


,student_id,student_name,department,attendance,study_hours,assignments_completed,internal_marks,final_marks
36,STU037,Veer Saxena,Mechanical,75.0,3.0,6.0,30,67.0
37,STU038,Zoya Hussain,Mechanical,88.0,4.5,8.0,35,80.0
38,STU039,Manav Trivedi,Mechanical,69.0,2.0,5.0,27,59.0
39,STU040,Ira Fernandes,Mechanical,84.0,4.0,7.0,33,76.0
40,STU007,Vivaan Joshi,Computer Science,84.0,4.0,7.0,34,78.0


## 4. Understand the Structure

These checks describe the table before any cleaning:

- `shape` gives the number of rows and columns
- `columns` lists the field names
- `dtypes` shows the type stored in each column
- `info()` combines the row count, types, and how many values are present


In [5]:
print("Shape (rows, columns):", df.shape)
print()
print("Columns:")
print(df.columns)
print()
print("Data types:")
print(df.dtypes)
print()
df.info()


Shape (rows, columns): (41, 8)

Columns:
Index(['student_id', 'student_name', 'department', 'attendance', 'study_hours',
       'assignments_completed', 'internal_marks', 'final_marks'],
      dtype='str')

Data types:
student_id                   str
student_name                 str
department                   str
attendance               float64
study_hours              float64
assignments_completed    float64
internal_marks             int64
final_marks              float64
dtype: object

<class 'pandas.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   student_id             41 non-null     str    
 1   student_name           41 non-null     str    
 2   department             41 non-null     str    
 3   attendance             39 non-null     float64
 4   study_hours            40 non-null     float64
 5   assignments_completed  40 non-null     flo

## 5. Check Missing Values

We check for missing values so incomplete records are visible before averages and charts are calculated. A number above zero means that column has blank cells.


In [6]:
df.isnull().sum()


student_id               0
student_name             0
department               0
attendance               2
study_hours              1
assignments_completed    1
internal_marks           0
final_marks              1
dtype: int64

## 6. Handle Missing Values

Blank cells are not all treated the same way.

- **Attendance, study hours, and assignments** are filled with the **median**. The median is the middle value, so one unusual student does not pull the fill as far as the mean would. These students still have the rest of their record, including final marks, so keeping them is more useful than deleting them.
- **Final marks** are different. That score is the result we are reporting. Filling it in would invent an exam result, so the row with a missing final mark is **dropped**.


In [7]:
attendance_median = df["attendance"].median()
study_hours_median = df["study_hours"].median()
assignments_median = df["assignments_completed"].median()

print("Median attendance:", attendance_median)
print("Median study hours:", study_hours_median)
print("Median assignments completed:", assignments_median)

df["attendance"] = df["attendance"].fillna(attendance_median)
df["study_hours"] = df["study_hours"].fillna(study_hours_median)
df["assignments_completed"] = df["assignments_completed"].fillna(assignments_median)

rows_before_drop = len(df)
df = df.dropna(subset=["final_marks"]).copy()
print("Rows removed because final marks were missing:", rows_before_drop - len(df))
print()
print("Missing values after cleaning:")
print(df.isnull().sum())


Median attendance: 84.0
Median study hours: 4.0
Median assignments completed: 7.0
Rows removed because final marks were missing: 1

Missing values after cleaning:
student_id               0
student_name             0
department               0
attendance               0
study_hours              0
assignments_completed    0
internal_marks           0
final_marks              0
dtype: int64


## 7. Check Duplicate Records

`duplicated()` marks a row when an earlier row is exactly the same. `sum()` counts how many copies were found. The raw file includes one repeated student row.


In [8]:
print("Duplicate rows:", df.duplicated().sum())
df[df.duplicated(keep=False)]


Duplicate rows: 1


,student_id,student_name,department,attendance,study_hours,assignments_completed,internal_marks,final_marks
6,STU007,Vivaan Joshi,Computer Science,84.0,4.0,7.0,34,78.0
40,STU007,Vivaan Joshi,Computer Science,84.0,4.0,7.0,34,78.0


## 8. Remove Duplicates

`drop_duplicates()` keeps the first copy and removes the repeated row. Leaving the duplicate in would count Vivaan Joshi twice and slightly change the averages.


In [9]:
df = df.drop_duplicates().copy()
print("Duplicate rows after removal:", df.duplicated().sum())
print("Rows remaining:", len(df))
df.head()


Duplicate rows after removal: 0
Rows remaining: 39


,student_id,student_name,department,attendance,study_hours,assignments_completed,internal_marks,final_marks
0,STU001,Aarav Shah,Computer Science,92.0,5.5,8.0,38,86.0
1,STU002,Diya Patel,Computer Science,88.0,4.5,8.0,36,82.0
2,STU003,Kabir Mehta,Computer Science,95.0,6.0,9.0,40,91.0
3,STU004,Ananya Iyer,Computer Science,81.0,3.5,7.0,32,74.0
4,STU005,Rohan Desai,Computer Science,76.0,3.0,6.0,30,68.0


## 9. Filter the Data

Filtering keeps only the rows that match a condition. These three views answer common academic questions: who meets the attendance rule, who scored 70 or above, and who studies at least 3 hours.


In [10]:
attendance_ok = df[df["attendance"] >= 75]
print("Students with attendance >= 75:", len(attendance_ok))
attendance_ok[["student_name", "department", "attendance", "final_marks"]].head()


Students with attendance >= 75: 30


,student_name,department,attendance,final_marks
0,Aarav Shah,Computer Science,92.0,86.0
1,Diya Patel,Computer Science,88.0,82.0
2,Kabir Mehta,Computer Science,95.0,91.0
3,Ananya Iyer,Computer Science,81.0,74.0
4,Rohan Desai,Computer Science,76.0,68.0


In [11]:
strong_scores = df[df["final_marks"] >= 70]
print("Students with final marks >= 70:", len(strong_scores))
strong_scores[["student_name", "department", "final_marks"]].head()


Students with final marks >= 70: 27


,student_name,department,final_marks
0,Aarav Shah,Computer Science,86.0
1,Diya Patel,Computer Science,82.0
2,Kabir Mehta,Computer Science,91.0
3,Ananya Iyer,Computer Science,74.0
5,Meera Nair,Computer Science,85.0


In [12]:
steady_study = df[df["study_hours"] >= 3]
print("Students with study hours >= 3:", len(steady_study))
steady_study[["student_name", "study_hours", "final_marks"]].head()


Students with study hours >= 3: 31


,student_name,study_hours,final_marks
0,Aarav Shah,5.5,86.0
1,Diya Patel,4.5,82.0
2,Kabir Mehta,6.0,91.0
3,Ananya Iyer,3.5,74.0
4,Rohan Desai,3.0,68.0


## 10. Sort the Data

Sorting changes the order of rows without changing the values. Sorting by final marks, highest first, puts the strongest results at the top. Sorting by attendance does the same for presence in class.


In [13]:
df.sort_values("final_marks", ascending=False)[
    ["student_name", "department", "final_marks", "attendance"]
].head(8)


,student_name,department,final_marks,attendance
8,Arjun Reddy,Computer Science,94.0,97.0
2,Kabir Mehta,Computer Science,91.0,95.0
18,Yash Agarwal,Information Technology,88.0,94.0
23,Lavanya Krishnan,Electronics,87.0,93.0
0,Aarav Shah,Computer Science,86.0,92.0
5,Meera Nair,Computer Science,85.0,90.0
33,Anika Paul,Mechanical,84.0,90.0
12,Aditya Rao,Information Technology,84.0,91.0


In [14]:
df.sort_values("attendance", ascending=False)[
    ["student_name", "department", "attendance", "final_marks"]
].head(8)


,student_name,department,attendance,final_marks
8,Arjun Reddy,Computer Science,97.0,94.0
2,Kabir Mehta,Computer Science,95.0,91.0
18,Yash Agarwal,Information Technology,94.0,88.0
23,Lavanya Krishnan,Electronics,93.0,87.0
0,Aarav Shah,Computer Science,92.0,86.0
27,Saanvi Mishra,Electronics,91.0,83.0
12,Aditya Rao,Information Technology,91.0,84.0
5,Meera Nair,Computer Science,90.0,85.0


## 11. Basic Statistical Analysis

These five functions describe final marks in one pass:

- `mean()` is the average
- `median()` is the middle score
- `min()` and `max()` are the lowest and highest scores
- `std()` is the standard deviation, which shows how spread out the scores are


In [15]:
print("Mean final marks:", round(df["final_marks"].mean(), 2))
print("Median final marks:", df["final_marks"].median())
print("Minimum final marks:", df["final_marks"].min())
print("Maximum final marks:", df["final_marks"].max())
print("Standard deviation:", round(df["final_marks"].std(), 2))


Mean final marks: 74.56
Median final marks: 76.0
Minimum final marks: 55.0
Maximum final marks: 94.0
Standard deviation: 9.87


## 12. Summary with describe()

`describe()` prints count, mean, standard deviation, minimum, quartiles, and maximum for every numeric column. It is a quick health check of the cleaned table.


In [16]:
df.describe()


,attendance,study_hours,assignments_completed,internal_marks,final_marks
count,39.000000,39.000000,39.000000,39.000000,39.000000
mean,82.179487,3.884615,6.948718,32.846154,74.564103
std,8.662903,1.183558,1.255487,3.808152,9.872329
min,64.000000,2.000000,4.000000,25.000000,55.000000
25%,75.500000,3.000000,6.000000,30.000000,67.500000
50%,84.000000,4.000000,7.000000,33.000000,76.000000
75%,88.500000,4.500000,8.000000,36.000000,82.000000
max,97.000000,6.500000,9.000000,40.000000,94.000000


## 13. Grouping and Aggregation

`groupby()` splits students by department and then calculates a summary for each group. The first example is the average final mark. The second example counts students and also finds the minimum and maximum final mark in each department.


In [17]:
department_average = df.groupby("department")["final_marks"].mean().sort_values(ascending=False)
department_average


department
Computer Science          79.100000
Electronics               74.400000
Information Technology    74.200000
Mechanical                70.111111
Name: final_marks, dtype: float64

In [18]:
department_summary = df.groupby("department").agg(
    students=("student_id", "count"),
    average_final_marks=("final_marks", "mean"),
    lowest_final_marks=("final_marks", "min"),
    highest_final_marks=("final_marks", "max"),
    average_attendance=("attendance", "mean"),
)
department_summary


,students,average_final_marks,lowest_final_marks,highest_final_marks,average_attendance
department,,,,,
Computer Science,10,79.100000,61.0,94.0,85.200000
Electronics,10,74.400000,60.0,87.0,82.200000
Information Technology,10,74.200000,58.0,88.0,82.600000
Mechanical,9,70.111111,55.0,84.0,78.333333


## 14. Derived Insights

These calculations turn the cleaned table into a few sentences a faculty member or student advisor can use: overall averages, the highest-performing student, the strongest department, and the share of students who meet the 75% attendance rule.


In [19]:
average_final = df["final_marks"].mean()
average_attendance = df["attendance"].mean()

top_student = df.loc[df["final_marks"].idxmax()]
best_department = department_average.idxmax()
attendance_rate = (df["attendance"] >= 75).mean() * 100

print("Students in the cleaned dataset:", len(df))
print("Average final marks:", round(average_final, 2))
print("Average attendance:", round(average_attendance, 2))
print(
    "Highest-performing student:",
    top_student["student_name"],
    "(",
    int(top_student["final_marks"]),
    "marks,",
    top_student["department"],
    ")",
)
print("Department with the highest average:", best_department)
print("Students meeting 75% attendance:", round(attendance_rate, 1), "%")


Students in the cleaned dataset: 39
Average final marks: 74.56
Average attendance: 82.18
Highest-performing student: Arjun Reddy ( 94 marks, Computer Science )
Department with the highest average: Computer Science
Students meeting 75% attendance: 76.9 %


## 15. Simple Visualizations

Two charts make the same story easier to see.

1. A bar chart of average final marks by department.
2. A scatter plot of attendance against final marks.

Both charts are saved under `notebooks/figures/` as well as shown here.


In [20]:
figures = root / "notebooks" / "figures"
figures.mkdir(parents=True, exist_ok=True)

department_average.sort_values().plot(kind="barh", color="#2563EB", figsize=(8, 4))
plt.title("Average Final Marks by Department")
plt.xlabel("Average final marks")
plt.ylabel("")
plt.xlim(0, 100)
plt.tight_layout()
plt.savefig(figures / "average_marks_by_department.png", dpi=120)
plt.show()


/var/folders/82/_ng802y92zv72h2wzqb4955m0000gn/T/ipykernel_18442/2313144630.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [21]:
plt.figure(figsize=(8, 5))
plt.scatter(df["attendance"], df["final_marks"], color="#7C3AED", alpha=0.85)
plt.title("Attendance vs Final Marks")
plt.xlabel("Attendance (%)")
plt.ylabel("Final marks")
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(figures / "attendance_vs_final_marks.png", dpi=120)
plt.show()


/var/folders/82/_ng802y92zv72h2wzqb4955m0000gn/T/ipykernel_18442/408494874.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 16. Save Processed Data for the Website

A browser cannot run Pandas. This cell writes the cleaned records and the summary numbers to JSON. The CampusCare home page reads that file.

The same JSON is copied into `static/data/` so Django can serve it without an extra API.


In [22]:
records = df.copy()
number_columns = [
    "attendance",
    "study_hours",
    "assignments_completed",
    "internal_marks",
    "final_marks",
]
for column in number_columns:
    records[column] = records[column].apply(
        lambda value: int(value) if float(value).is_integer() else round(float(value), 2)
    )

processed = {
    "pipeline": "CSV → Pandas analysis → processed JSON → CampusCare landing page",
    "source_csv": "data/student_performance.csv",
    "cleaning_notes": {
        "missing_attendance_filled_with_median": float(attendance_median),
        "missing_study_hours_filled_with_median": float(study_hours_median),
        "missing_assignments_filled_with_median": float(assignments_median),
        "rows_removed_missing_final_marks": 1,
        "duplicate_rows_removed": 1,
    },
    "summary": {
        "total_students": int(len(records)),
        "average_attendance": round(float(average_attendance), 2),
        "average_final_marks": round(float(average_final), 2),
        "top_performer": {
            "student_name": top_student["student_name"],
            "department": top_student["department"],
            "final_marks": int(top_student["final_marks"]),
        },
        "best_department": best_department,
        "attendance_criteria_percent": round(float(attendance_rate), 1),
    },
    "records": records.to_dict(orient="records"),
}

output_paths = [
    root / "data" / "student_performance_processed.json",
    root / "static" / "data" / "student_performance_processed.json",
]
for output_path in output_paths:
    output_path.write_text(json.dumps(processed, indent=2))
    print("Saved", output_path)


Saved /Users/hirenrathva/Desktop/CampusCare/data/student_performance_processed.json
Saved /Users/hirenrathva/Desktop/CampusCare/static/data/student_performance_processed.json


## Key Insights

- The raw file has 41 rows. After cleaning there are **39 students**: one duplicate row was removed, and one row was removed because final marks were missing.
- Missing attendance, study hours, and assignments were filled with the median (**84%**, **4 hours**, and **7 assignments**) so those students could stay in the analysis.
- Average final marks are **74.56**, and average attendance is **82.18%**.
- The highest-performing student is **Arjun Reddy** (Computer Science) with **94** final marks.
- **Computer Science** has the highest department average (**79.10**), followed by Electronics, Information Technology, and Mechanical.
- **76.9%** of students meet the 75% attendance rule.
- Students with higher attendance generally also have higher final marks, which is why the scatter plot slopes upward.
